# US Bureau Census

This notebook is a testing of the how we will grab data from US Bureau Census for population data

In [1]:
import requests
import pandas as pd
import duckdb
import pyarrow as pa
from dotenv import load_dotenv
import os

from pipeline.dbloader import DBLoader

In [2]:
load_dotenv()
CENSUS_KEY = os.getenv("CENSUS_API_KEY")

In [3]:
def fetch_census_population(years: list[int]) -> duckdb.DuckDBPyRelation:
    """
    Fetches US national population estimates from Census PEP API.
    Covers 2000-2009 (intercensal), 2010-2019, and 2020-present as separate vintages.
    """
    all_records = []

    # --- Vintage ranges — Census restructures endpoints per decade ---
    vintage_map = {
        2000: range(2000, 2010),   # intercensal estimates
        2010: range(2010, 2020),   # 2010s vintage
        2019: range(2020, 2026),   # 2020s vintage (use 2019 base endpoint)
    }

    for vintage, year_range in vintage_map.items():
        target_years = [y for y in years if y in year_range]
        if not target_years:
            continue

        if vintage == 2000:
            # Intercensal 2000-2009 endpoint
            url = f"https://api.census.gov/data/2000/pep/int_population"
            params = {
                "get": "POP,DATE_DESC",
                "for": "us:1",
                "key": CENSUS_KEY,
            }
        elif vintage == 2010:
            url = f"https://api.census.gov/data/2019/pep/population"
            params = {
                "get": "POP,DATE_CODE,DATE_DESC",
                "for": "us:1",
                "key": CENSUS_KEY,
            }
        else:
            # 2020s use charv endpoint
            url = f"https://api.census.gov/data/2023/pep/population"
            params = {
                "get": "POP_2020,POP_2021,POP_2022,POP_2023",
                "for": "us:1",
                "key": CENSUS_KEY,
            }

        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            headers = data[0]
            rows    = data[1:]
            for row in rows:
                all_records.append(dict(zip(headers, row)))
            print(f"[INFO] Fetched vintage {vintage}: {len(rows)} rows")
        except requests.RequestException as e:
            print(f"[ERROR] Vintage {vintage}: {e}")

    if not all_records:
        return duckdb.sql("SELECT 1 WHERE FALSE")

    arrow = pa.Table.from_pylist(all_records)
    return duckdb.sql("SELECT * FROM arrow")

In [10]:
def fetch_population_clean() -> pd.DataFrame:
    """
    Simplified single-call fetch for national US population 2010-2023.
    Returns a clean DataFrame with year and population columns.
    """
    url = "https://api.census.gov/data/2019/pep/population"
    params = {
        "get":  "POP,DATE_CODE,DATE_DESC",
        "for":  "us:1",
        "key":  CENSUS_KEY,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data    = r.json()
    headers = data[0]
    rows    = data[1:]

    df = pd.DataFrame(rows, columns=headers)
    df["POP"]       = pd.to_numeric(df["POP"])
    df["DATE_CODE"] = pd.to_numeric(df["DATE_CODE"])

    # DATE_CODE 1 = April 2010 census base, 2+ = July 1 estimates per year
    df = df[df["DATE_CODE"] >= 2].copy()
    df["year"] = 1998 + df["DATE_CODE"]   # DATE_CODE 2 = 2010, 3 = 2011 ...

    return df[["year", "POP"]].rename(columns={"POP": "population"})

In [11]:
# --- Run it ---
pop_df = fetch_population_clean()
print(pop_df)

# --- Write to PostgreSQL using your existing function ---
pop_rel = duckdb.sql("SELECT * FROM pop_df")

    year  population
1   2000   308758105
2   2001   309321666
3   2002   311556874
4   2003   313830990
5   2004   315993715
6   2005   318301008
7   2006   320635163
8   2007   322941311
9   2008   324985539
10  2009   326687501
11  2010   328239523


In [ ]:
write_to_postgres(
    relation=pop_rel,
    table_name="t_us_population",
    schema={
        "year":       ("INTEGER", True),
        "population": ("BIGINT",  False),
    },
    engine=engine,
    pg_schema="bronze"
)